# 02 - Synthetic fixture and donor split

This notebook is updated by more than one commit. Each commit adds a dated section.

| Section | Commit | Date |
|---|---|---|
| C03 synthetic fixture | `test: add deterministic synthetic multimodal fixtures` | 2026-07-27 |

## C03 - Deterministic synthetic fixture (2026-07-27)

- **Meeting item:** Prototype validation
- **Commit:** C03 - `test: add deterministic synthetic multimodal fixtures`
- **Commit achievement:** creates a deterministic two-view test world with donor nuisance.
- **Data mode:** `synthetic`. Every array below is generated in memory; nothing is read from disk.
- **Evidence labels used:** verified, unknown
- **Human gate:** does the same seed reproduce identical arrays while a different seed changes them?
- **Stop condition:** NaN or infinity, missing identifiers, biological names, or nondeterminism.

All generation logic lives in `src/p22/testing/synthetic.py`.

In [ ]:
import numpy as np
import pandas as pd

import p22
from p22.testing import make_synthetic_multimodal

dataset = make_synthetic_multimodal(seed=42)

print(f"data mode: {dataset.generation['data_mode']}")
print(f"generator: {dataset.generation['generator']}")
print(f"view_a: {dataset.view_a.shape} {dataset.view_a.dtype}")
print(f"view_b: {dataset.view_b.shape} {dataset.view_b.dtype}")
print(f"cells: {dataset.n_cells}, donors: {dataset.n_donors}, classes: {dataset.n_classes}")
print(f"finite view_a: {bool(np.isfinite(dataset.view_a).all())}")
print(f"finite view_b: {bool(np.isfinite(dataset.view_b).all())}")
dataset.metadata.head()

### Label mix and donor structure (evidence label: verified)

Class balance is reported, never promised. Donors carry their own class mix, which is why
donor-blind evaluation would flatter a model.

In [ ]:
print("cells per class:")
print(dataset.label_distribution().to_string())
print("")
print("donor by class cell counts:")
print(dataset.donor_label_counts().to_string())

In [ ]:
donor_summary = (
    dataset.metadata.groupby("donor_id")[["view_a_total", "view_b_total"]]
    .agg(["mean", "std"])
    .round(3)
)
spread_between = dataset.metadata.groupby("donor_id")["view_b_total"].mean().std()
spread_within = dataset.metadata.groupby("donor_id")["view_b_total"].std().mean()
print(f"view_b donor-mean spread between donors: {spread_between:.3f}")
print(f"view_b mean spread within a donor: {spread_within:.3f}")
print("donor shift is present by construction; view B carries a direct donor offset")
donor_summary

### Determinism (evidence label: verified)

In [ ]:
repeat_same = make_synthetic_multimodal(seed=42)
other_seed = make_synthetic_multimodal(seed=43)

checks = {
    "same seed, view_a identical": np.array_equal(dataset.view_a, repeat_same.view_a),
    "same seed, view_b identical": np.array_equal(dataset.view_b, repeat_same.view_b),
    "same seed, labels identical": np.array_equal(dataset.labels, repeat_same.labels),
    "different seed, view_a changes": not np.array_equal(dataset.view_a, other_seed.view_a),
    "different seed, labels change": not np.array_equal(dataset.labels, other_seed.labels),
    "donor identifiers stable across seeds": np.array_equal(
        dataset.donor_ids, other_seed.donor_ids
    ),
}
print(pd.Series(checks).to_string())
assert all(checks.values()), checks

### Why the signal is imperfect (evidence label: verified)

Each view reads a different half of the latent space strongly and the other half weakly, so
neither view alone contains the whole label signal. Model fitting is not done here; that
starts in C07 and C10.

In [ ]:
weights = dataset.generation["view_latent_weights"]
print(f"latent weights, view_a: {weights['view_a']}")
print(f"latent weights, view_b: {weights['view_b']}")
print(f"class signal: {dataset.generation['class_signal']}")
print(f"donor nuisance: {dataset.generation['donor_nuisance']}")
print(f"latent noise: {dataset.generation['noise']}")
print("")
per_class = (
    pd.DataFrame(dataset.view_a[:, :4], columns=[f"view_a_f{i}" for i in range(4)])
    .assign(label_name=dataset.metadata["label_name"].to_numpy())
    .groupby("label_name")
    .mean()
    .round(3)
)
print("per-class feature means overlap, so classes are not separable by inspection:")
per_class

In [ ]:
import re

assert dataset.generation["data_mode"] == p22.SYNTHETIC
assert np.isfinite(dataset.view_a).all() and np.isfinite(dataset.view_b).all()
assert dataset.metadata.notna().all().all()
assert len(set(dataset.cell_ids)) == dataset.n_cells
assert set(np.unique(dataset.labels)) == set(range(dataset.n_classes))
assert all(re.fullmatch(r"donor_\d+", value) for value in dataset.donor_ids)
assert all(re.fullmatch(r"cell_\d+", value) for value in dataset.cell_ids)
print("all C03 gate checks passed")

### Unknown (evidence label: unknown)

- Whether this synthetic world resembles any real dataset. It is a controlled test world, not a simulation of one.
- What accuracy is reachable here; that is measured in C10 with donor-held-out splits and five seeds.

## Gate

**PASS**

- **Observed (C03):** seed 42 produces a 240-cell, 12-donor, 3-class world with `view_a` of shape (240, 16) and `view_b` of shape (240, 12), no NaN or infinity, unique neutral identifiers, uneven per-donor class mix, a donor offset in view B, identical arrays on a repeated seed, and different arrays on a different seed.
- **Remains unknown:** reachable accuracy in this world and how closely it resembles any real dataset.
- **Next decision owner:** researcher continues with C04, which adds donor-held-out splitting to this same notebook.